In [22]:
import pandas as pd
import numpy as np
from pathlib import Path

In [23]:
def find_files(folder: Path, extension: str) -> list[Path]:
    return [f for f in folder.iterdir() if f.is_file() and f.suffix == extension]

def downsample(fileData: np.lib.npyio.NpzFile, delay: float) -> pd.DataFrame:
    t = fileData["t"]
    freq = fileData["freq"]
    # Downsample the data by selecting only the samples that are at least `delay` seconds apart
    downsampledIndices = [0]  # Always include the first sample
    lastTime = t[0]
    for i in range(1, len(t)):
        if t[i] - lastTime >= delay:
            downsampledIndices.append(i)
            lastTime = t[i]
    return pd.DataFrame({
        "t": t[downsampledIndices],
        "freq": freq[downsampledIndices]
    })


In [24]:
dataFolder = Path("./data").absolute()
resultsFolder = Path("./results").absolute()
classToFolders = {
    "direct": ["case_iec104_direct_percmd"],
    "delay": ["case_iec104_mitm_delay_dmin0p4_dmax0p6_percmd", "case_iec104_mitm_delay_dmin0p15_dmax0p35_percmd", "case_iec104_mitm_delay_dmin0p65_dmax0p85_percmd"],
    "drop": ["case_iec104_mitm_drop_pdrop0p1_percmd", "case_iec104_mitm_drop_pdrop0p5_percmd", "case_iec104_mitm_drop_pdrop0p9_percmd"],
    "mod": ["case_iec104_mitm_mod_pmod0p1_f0p1_percmd", "case_iec104_mitm_mod_pmod0p5_f0p1_percmd", "case_iec104_mitm_mod_pmod0p9_f0p1_percmd"]
}
classToFoldersPaths = {c: [resultsFolder / f for f in folders] for c, folders in classToFolders.items()}
# Find all files with ".npz" extension in the folders and associate it with the class
classToFilesPaths = {c: [f for folder in folders for f in find_files(folder, ".npz")] for c, folders in classToFoldersPaths.items()}

In [25]:
# Load the data from the .npz files and create a DataFrame
classToData = {}

for c, files in classToFilesPaths.items():
    classDataDf = [] 
    for filePath in files:
        df = np.load(filePath)
        classDataDf.append(df)
    classToData[c] = classDataDf

In [26]:
# Create the final dataframe
tripTime = 1 # The trip event happened at 1 second
finalDf = pd.DataFrame(columns=["t-from-trip", "freq", "run-id", "class"])
runId = 0
for c, dataList in classToData.items():
    for data in dataList:
        runId += 1
        dataDf = downsample(data, delay=0.0001)
        dataDf["t-from-trip"] = dataDf["t"] - tripTime
        dataDf["class"] = c
        dataDf["run-id"] = runId
        # Remove the rows where t-from-trip is negative (before the trip event)
        dataDf = dataDf[dataDf["t-from-trip"] >= 0]
        finalDf = pd.concat([finalDf, dataDf[["t-from-trip", "freq", "run-id", "class"]]], ignore_index=True)

/tmp/ipykernel_4008508/1359888980.py:14: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  finalDf = pd.concat([finalDf, dataDf[["t-from-trip", "freq", "run-id", "class"]]], ignore_index=True)


In [27]:
finalDf.head()

,t-from-trip,freq,run-id,class
0,0.000100,59.999947,1,direct
1,0.033433,59.969277,1,direct
2,0.066767,59.946259,1,direct
3,0.099900,59.927761,1,direct
4,0.100100,59.927659,1,direct


In [28]:
finalDf.to_csv(dataFolder / "final_data.csv", index=False)